# Agentic Protocol Engineering: The Wiring Harness of Trust

## Chapter 8 Presentation

**When Arrows Bleed: Building Trust in Agentic Systems**

*Austin tapped the whiteboard... "Every time we sketch the Agentic Stack... the arrows start to bleed."*

This notebook demonstrates the key protocols that prevent trust from leaking through the seams of autonomous systems.

![Maturity Ladder](https://via.placeholder.com/800x400?text=Maturity+Ladder+Diagram)

*Run each cell to see protocols in action!*

## What Are Agentic Protocols?

**APIs connect components. Protocols connect trust.**

Traditional APIs describe *how* to call a method. Agentic protocols define:
- ✅ What must travel with every handoff (evidence, metadata, approvals)
- ✅ How handoffs are validated before acceptance
- ✅ What happens when validation fails

**The Four Core Protocols:**
1. **MCP** (Model Context Protocol) - Stable context for models
2. **ACP** (Agent Communication Protocol) - Structured agent messaging  
3. **A2A** (Agent-to-Agent Protocol) - Multi-agent coordination
4. **ANP** (Agent Network Protocol) - Cross-organizational trust

Plus emerging candidates: Retrieval Provenance, Memory Access, Action Governance, Tool Safety.

In [ ]:
# Import Required Libraries
import json
import time
from datetime import datetime
from typing import Dict, Any, List
import hashlib

## Protocol Base Classes

All protocols inherit from a common base that provides:
- **Validation**: Ensures protocol structure is correct
- **Serialization**: Converts to JSON for transmission
- **Type Safety**: Uses Python typing for reliability

In [ ]:
class ProtocolBase:
    """Base class for all protocols with common validation and serialization."""
    def validate(self) -> bool:
        """Validate the protocol structure."""
        raise NotImplementedError

    def to_json(self) -> str:
        """Serialize to JSON."""
        return json.dumps(self.__dict__, indent=2, default=str)

## MCP: Model Context Protocol

**Stabili zes model inputs against context drift**

- Versioned context structure
- Predictable slots for different data types
- Token budget management
- Guards against stale or incomplete inputs

*Example: Every reasoning cycle gets consistent, validated context*

In [ ]:
class MCPContext(ProtocolBase):
    """Model Context Protocol - Stable, versioned context packaging."""
    def __init__(self, context_id: str, version: str = "1.0"):
        self.context_id = context_id
        self.version = version
        self.context_slots: List[Dict[str, Any]] = []
        self.budget_tokens = 1000

    def add_slot(self, slot_name: str, content: str):
        """Add a context slot with content."""
        self.context_slots.append({
            "slot": slot_name,
            "content": content
        })

    def validate(self) -> bool:
        return bool(self.context_id and self.context_slots)

## Retrieval Provenance Protocol

**Evidence inseparable from data**

- Origin tracking (where did this come from?)
- Freshness validation (when was it retrieved?)
- Trust scoring (how reliable is the source?)
- Cryptographic signatures (has it been tampered with?)

*Without provenance, "trusted information" has no verifiable trail*

In [ ]:
class RetrievalProvenance(ProtocolBase):
    """Retrieval Provenance Protocol - Evidence and provenance for retrieved data."""
    def __init__(self, document_id: str, origin: str, content: str):
        self.document_id = document_id
        self.origin = origin
        self.retrieved_at = datetime.now()
        self.trust_score = 0.95
        self.signature = self._generate_signature(content)
        self.content = content

    def _generate_signature(self, content: str) -> str:
        """Simple signature simulation."""
        return hashlib.sha256(content.encode()).hexdigest()[:16]

    def validate(self) -> bool:
        return bool(self.document_id and self.origin and self.content)

## ACP: Agent Communication Protocol

**Structured messaging that agents can actually parse**

- Unique message IDs and session tracking
- Explicit sender/receiver identification
- Conversation threading
- Timestamped for audit trails

*Where APIs leave no memory of conversation, ACP leaves a ledger*

In [ ]:
class ACPMessage(ProtocolBase):
    """Agent Communication Protocol - Structured message envelope."""
    def __init__(self, message_id: str, msg_type: str, session_id: str,
                 sender: str, receiver: str, payload: Dict[str, Any]):
        self.message_id = message_id
        self.type = msg_type
        self.session_id = session_id
        self.from_agent = sender
        self.to_agent = receiver
        self.payload = payload
        self.timestamp = datetime.now()

    def validate(self) -> bool:
        required = [self.message_id, self.type, self.session_id,
                   self.from_agent, self.to_agent, self.payload]
        return all(required)

## Sandbox Protocol

**Safety declarations for tool execution**

- Execution classes: safe, restricted, privileged
- Resource limits and network controls
- Runtime validation before execution
- Prevents privilege escalation

*Turns "don't do that" into "can't do that"*

In [ ]:
class SandboxProtocol(ProtocolBase):
    """Sandbox Protocol - Execution class declarations for safety."""
    def __init__(self, tool: str, execution_class: str, requirements: Dict[str, Any]):
        self.tool = tool
        self.execution_class = execution_class  # 'safe', 'restricted', 'privileged'
        self.requirements = requirements

    def validate(self) -> bool:
        valid_classes = ['safe', 'restricted', 'privileged']
        return (self.tool and self.execution_class in valid_classes and
                isinstance(self.requirements, dict))

## Agent Class

**Protocol-compliant agents that enforce trust**

- Message sending/receiving with ACP
- Context management with MCP
- Task execution with Sandbox validation
- Memory for state persistence

*Agents that speak the same protocol language*

In [ ]:
class Agent:
    """Simple agent that uses protocols for communication."""
    def __init__(self, name: str):
        self.name = name
        self.memory: Dict[str, Any] = {}

    def send_message(self, receiver: 'Agent', msg_type: str, payload: Dict[str, Any],
                    session_id: str = "session_001") -> ACPMessage:
        """Send a structured message using ACP."""
        message = ACPMessage(
            message_id=f"msg_{int(time.time())}_{self.name}",
            msg_type=msg_type,
            session_id=session_id,
            sender=self.name,
            receiver=receiver.name,
            payload=payload
        )
        if message.validate():
            print(f"📤 {self.name} → {receiver.name}: {msg_type}")
            print(message.to_json())
            receiver.receive_message(message)
            return message
        else:
            raise ValueError("Invalid message structure")

    def receive_message(self, message: ACPMessage):
        """Process incoming ACP message."""
        if not message.validate():
            print(f"❌ {self.name} rejected invalid message")
            return

        print(f"📥 {self.name} received: {message.type}")
        # Process based on message type
        if message.type == "task_request":
            self.handle_task_request(message.payload)
        elif message.type == "context_update":
            self.handle_context_update(message.payload)

    def handle_task_request(self, payload: Dict[str, Any]):
        """Handle a task request with sandbox protocol."""
        tool = payload.get("tool", "unknown")
        sandbox = SandboxProtocol(
            tool=tool,
            execution_class=payload.get("execution_class", "safe"),
            requirements=payload.get("requirements", {})
        )

        if sandbox.validate():
            print(f"🔒 {self.name} executing {tool} in {sandbox.execution_class} mode")
            # Simulate execution
            result = f"Executed {tool} successfully"
            self.memory["last_result"] = result
        else:
            print(f"🚫 {self.name} rejected unsafe tool execution")

    def handle_context_update(self, payload: Dict[str, Any]):
        """Handle context update with MCP."""
        context = MCPContext(
            context_id=payload.get("context_id", "ctx_001"),
            version=payload.get("version", "1.0")
        )

        for slot in payload.get("context_slots", []):
            context.add_slot(slot["slot"], slot["content"])

        if context.validate():
            print(f"🧠 {self.name} updated context: {context.context_id}")
            self.memory["current_context"] = context
        else:
            print(f"❌ {self.name} rejected invalid context")

## Demo 1: Retrieval with Provenance

**Evidence travels with the data**

*Scenario: Retrieving medical dosage guidelines from a policy database*

The protocol ensures every retrieved fact carries its origin, timestamp, trust score, and cryptographic signature.

In [ ]:
# Demo 1: Retrieval with Provenance
doc = RetrievalProvenance(
    document_id="doc_123",
    origin="https://intranet.policy/drug-guidelines",
    content="Dosage: 50mg daily"
)
print("Retrieved document with provenance:")
print(doc.to_json())

## Demo 2: Context Packaging with MCP

**Stable, predictable inputs for models**

*Scenario: Packaging retrieved knowledge and current task for a reasoning model*

The MCP ensures the model always knows where to find specific types of information.

In [ ]:
# Demo 2: Context Packaging with MCP
context = MCPContext("ctx_456", "1.1")
context.add_slot("retrieved_knowledge", doc.content)
context.add_slot("current_task", "Prepare safety review summary")
print("Packaged context:")
print(context.to_json())

## Demo 3: Agent Communication with ACP

**Structured messaging between agents**

*Scenario: ResearchBot sends context update to ContentWriter*

ACP ensures every message is traceable, typed, and session-aware.

In [ ]:
# Demo 3: Agent Communication with ACP
research_agent = Agent("ResearchBot")
content_agent = Agent("ContentWriter")

research_agent.send_message(
    content_agent,
    "context_update",
    {
        "context_id": context.context_id,
        "version": context.version,
        "context_slots": context.context_slots
    }
)

## Demo 4: Task Execution with Sandbox Protocol

**Safe tool execution with enforceable boundaries**

*Scenario: Requesting report generation with restricted execution class*

The sandbox protocol ensures tools run with appropriate security constraints.

In [ ]:
# Demo 4: Task Execution with Sandbox Protocol
research_agent.send_message(
    content_agent,
    "task_request",
    {
        "tool": "generate_report",
        "execution_class": "restricted",
        "requirements": {
            "network_access": False,
            "max_runtime_seconds": 60
        }
    }
)

## The Agentic Protocol Maturity Ladder

**From ad hoc exchanges to federated trust**

```mermaid
graph TD
    A[L0: Ad Hoc Exchanges<br/>No protocols, informal communication] --> B[L1: Containment<br/>Sandbox Protocols<br/>Execution safety enforced]
    B --> C[L2: Context Stability<br/>MCP + Retrieval Protocols<br/>Stable context with provenance]
    C --> D[L3a: Structured Communication<br/>ACP + Memory Access Protocol<br/>Predictable messaging and state]
    D --> E[L3b: Coordination & Governance<br/>A2A + Action Invocation Contract<br/>Aligned work with enforceable rules]
    E --> F[L4: Federated Trust<br/>ANP<br/>Cross-organizational interoperability]
    F --> G[L5: Fully Federated Protocol Mesh<br/>All protocols unified<br/>Adaptive trust fabric]

    style A fill:#ffcccc
    style B fill:#ffddaa
    style C fill:#ffffaa
    style D fill:#aaffaa
    style E fill:#aaaaff
    style F fill:#ffaaaa
    style G fill:#aaffff
```

**Key Insight:** Protocols carry not just data, but trust, evidence, and governance. They transform fragile arrows into enforceable contracts.

*Run all cells above to see the complete demo!*